# Barcode Assessment — Detect, Crop, Orient & Decode

Full zero-training pipeline on photos in `tagss/`:

| Phase | Step |
|-------|------|
| **1** | Grayscale → Scharr gradients → threshold → morphological close → contour filter |
| **1b** | **Tight rotated crop** via `minAreaRect` perspective warp (not axis-aligned bbox) |
| **2** | Deskew with `minAreaRect` + `warpAffine` if needed |
| **3** | Decode with **pyzbar** |

Outputs: `cropped/` (raw + oriented crops), `debug/` (masks & annotated detections)

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from barcode_detect import detect_barcode_regions, draw_detections, crop_barcode, process_folder

PROJECT_ROOT = Path(".").resolve()
INPUT_DIR = PROJECT_ROOT / "tagss"
OUTPUT_DIR = PROJECT_ROOT / "cropped"
DEBUG_DIR = PROJECT_ROOT / "debug"

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Debug:  {DEBUG_DIR}")

In [ ]:
# Preview detection + rotated cropping on one sample image
sample_path = sorted(INPUT_DIR.glob("*"))[0]
image = cv2.imread(str(sample_path))
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

regions, debug = detect_barcode_regions(image)
annotated = draw_detections(image, regions)
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0, 0].imshow(image_rgb)
axes[0, 0].set_title(f"Original\n{sample_path.name}")
axes[0, 1].imshow(debug["gradient"], cmap="gray")
axes[0, 1].set_title("Gradient magnitude")
axes[1, 0].imshow(debug["mask"], cmap="gray")
axes[1, 0].set_title("Threshold + morphology")
axes[1, 1].imshow(annotated_rgb)
axes[1, 1].set_title(f"Detections ({len(regions)})")

for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

# Show rotated perspective crops
if regions:
    fig, axes = plt.subplots(1, len(regions), figsize=(4 * len(regions), 4))
    if len(regions) == 1:
        axes = [axes]
    for ax, region in zip(axes, regions):
        crop_rgb = cv2.cvtColor(crop_barcode(image, region), cv2.COLOR_BGR2RGB)
        ax.imshow(crop_rgb)
        ax.set_title("Rotated crop", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Run full pipeline: detect → rotated crop → orient → decode
results = process_folder(INPUT_DIR, OUTPUT_DIR, DEBUG_DIR, decode=True)

summary_rows = []
decode_rows = []

for item in results:
    if "error" in item:
        summary_rows.append({"source": item["source"], "found": 0, "decoded": 0, "status": item["error"]})
        continue

    summary_rows.append({
        "source": item["source"],
        "found": item["count"],
        "decoded": item["decoded_count"],
        "status": "OK",
    })

    for bc in item["barcodes"]:
        decode_rows.append({
            "source": item["source"],
            "index": bc["index"],
            "type": bc["type"] or "—",
            "decoded": bc["text"] or "FAILED",
        })

summary_df = pd.DataFrame(summary_rows)
decode_df = pd.DataFrame(decode_rows)

total_found = summary_df["found"].sum()
total_decoded = summary_df["decoded"].sum()
print(f"Images: {len(results)}  |  Barcodes found: {total_found}  |  Decoded: {total_decoded}")
summary_df

In [ ]:
# All decoded values (one row per barcode crop)
decode_df

In [ ]:
# Per-image view: annotated photo + each crop with decode result
for item in results:
    if "error" in item or item["count"] == 0:
        continue

    barcodes = item["barcodes"]
    n = len(barcodes)
    fig = plt.figure(figsize=(max(12, 3 * n), 9))
    fig.suptitle(
        f"{item['source']}  —  {item['decoded_count']}/{item['count']} decoded",
        fontsize=12,
    )

    ax0 = plt.subplot2grid((2, n), (0, 0), colspan=n)
    ax0.imshow(cv2.cvtColor(item["annotated"], cv2.COLOR_BGR2RGB))
    ax0.set_title("Detections (label = decoded value)")
    ax0.axis("off")

    for i, bc in enumerate(barcodes):
        ax = plt.subplot2grid((2, n), (1, i))
        show_img = bc["decoded"].oriented_image if bc["decoded"] else bc["crop"]
        ax.imshow(cv2.cvtColor(show_img, cv2.COLOR_BGR2RGB))
        label = bc["text"] if bc["text"] else "FAILED"
        btype = bc["type"] or "—"
        ax.set_title(f"#{bc['index']} {btype}\n{label}", fontsize=9,
                     color="green" if bc["text"] else "red")
        ax.axis("off")

    plt.tight_layout()
    plt.show()